# Flagged SGA 2020 Anchors
This program allows users to investigate flagged anchors from a savefile generated via VI-user-training.ipynb. 

### To do:
Add residuals as viewing option.

Add reviewer process where the reviewer inputs their name and gives their verdict on who was right for each galaxy flagged.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import os
import glob # get path names
from pathlib import Path 

import pandas as pd
from astropy.table import Table
from astropy.io import fits

import astropy.visualization as vis # for image enhancements
from scipy.ndimage import gaussian_filter

import ipywidgets as widgets # for buttons
from IPython.display import display, clear_output # for display functions

# Load flagged galaxies
flag_path = Path("/pscratch/sd/q/qshimp/VI_training/savefiles/QuillanIMR_flagged.csv")
flagged_df = pd.read_csv(flag_path)
len(flagged_df)

70

In [2]:
# Functions from VI-user-training.ipynb

# Image enhancer
def stretch_band(band, mode="asinh"):
    band = np.nan_to_num(band)
    if mode == "asinh":
        norm = vis.ImageNormalize(band, interval=vis.PercentileInterval(99.5), stretch=vis.AsinhStretch())
    elif mode == "log":
        norm = vis.ImageNormalize(band, interval=vis.PercentileInterval(99.5), stretch=vis.LogStretch())
    else:
        norm = vis.ImageNormalize(band, interval=vis.PercentileInterval(99.5), stretch=vis.LinearStretch())
    return norm(band)


# Image loader
def load_image(path, stretch="asinh"):

    # open fits file
    with fits.open(path) as hdul:
        data = hdul[0].data.astype(float)

    # Assign image data
    img = np.transpose(data, (1,2,0))
    g = img[:,:,0]
    r = img[:,:,1]
    z = img[:,:,2]

    # Add image stretch
    g_str = stretch_band(g, stretch)
    r_str = stretch_band(r, stretch)
    z_str = stretch_band(z, stretch)

    # Combine band layers by depth and bind pixel values
    rgb = np.dstack([z_str, r_str, g_str])
    rgb = np.clip(rgb,0,1)

    # Combine bands for a grayscale image
    gray = (0.5*r_str + 0.3*z_str + 0.2*g_str)
    gray = np.clip(gray,0,1)

    # Use Lupton technique
    lupton = vis.make_lupton_rgb(z, r, g, stretch=0.5, Q=10)

    # Create Gaussian blur   
    smooth = gaussian_filter(gray, sigma=4)

    # Isolate fine details with unsharp mask
    unsharp = gray - smooth    
    unsharp -= unsharp.min()
    unsharp /= unsharp.max()

    return rgb, lupton, gray, unsharp, g_str, r_str, z_str

# Load legacy survey jpgs
def load_jpg(row):
    # Get galaxy id
    tid = row["ref_id"]

    patternM = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Model/{tid}_*.jpg"
    matchesM = glob.glob(patternM)
    patternR = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Residual/{tid}_*.jpg"
    matchesR = glob.glob(patternR)
    patternI = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Image/{tid}_*.jpg"
    matchesI = glob.glob(patternI)

    model = mpimg.imread(matchesM[0])
    residual = mpimg.imread(matchesR[0])
    image = mpimg.imread(matchesI[0])
 
    return model, residual, image

In [3]:
# Image display
image_out = widgets.Output()

# Flagged galaxy selector
index_box = widgets.IntText(value=0, min=0, max=len(flagged_df)-1, step=1, description="Galaxy", continuous_update=False)

# Create view selector
view_selector = widgets.Dropdown(
    options=[
        "RGB",
        "Lupton RGB",
        "Grayscale",
        "Unsharp Mask",
        "RGB + Grayscale",
        "RGB + Grayscale + Unsharp",
        "6-view",
        "g band",
        "r band",
        "z band"
    ],
    value="RGB + Grayscale + Unsharp",
    description="View"
)

stretch_selector = widgets.Dropdown(options=["asinh", "log", "linear"], value="asinh", description="Stretch")

In [4]:
# Function to display flagged galaxies
def display_flagged_galaxy(change=None):

    # Keep index in bounds
    i = index_box.value
    if i < 0 or i >= len(flagged_df):
        return

    # Get dataframe row
    row = flagged_df.iloc[i]

    # Load images
    path = row["path"]
    rgb, lupton, gray, unsharp, g, r, z = load_image(path, stretch=stretch_selector.value)
    model, residual, image = load_jpg(row)
    
    with image_out:
        image_out.clear_output(wait=True)
        view = view_selector.value

        if view == "RGB":

            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(rgb, origin="lower")
            ax.set_title("RGB")
            ax.axis("off")

        elif view == "Lupton RGB":

            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(lupton, origin="lower")
            ax.set_title("Lupton RGB")
            ax.axis("off")

        elif view == "Grayscale":

            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(gray, origin="lower", cmap="gray")
            ax.set_title("Grayscale")
            ax.axis("off")

        elif view == "Unsharp Mask":

            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(unsharp, origin="lower", cmap="gray")
            ax.set_title("Unsharp Mask")
            ax.axis("off")

        elif view == "g band":

            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(g, origin="lower", cmap="gray")
            ax.set_title("g band")
            ax.axis("off")

        elif view == "r band":

            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(r, origin="lower", cmap="gray")
            ax.set_title("r band")
            ax.axis("off")

        elif view == "z band":

            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(z, origin="lower", cmap="gray")
            ax.set_title("z band")
            ax.axis("off")

        elif view == "RGB + Grayscale":

            fig, axes = plt.subplots(1, 2, figsize=(10,5))

            axes[0].imshow(rgb, origin="lower")
            axes[0].set_title("RGB")

            axes[1].imshow(gray, origin="lower", cmap="gray")
            axes[1].set_title("Grayscale")

            for ax in axes:
                ax.axis("off")

        elif view == "RGB + Grayscale + Unsharp":

            fig, axes = plt.subplots(1, 3, figsize=(15,5))

            axes[0].imshow(rgb, origin="lower")
            axes[0].set_title("RGB")

            axes[1].imshow(gray, origin="lower", cmap="gray")
            axes[1].set_title("Grayscale")

            axes[2].imshow(unsharp, origin="lower", cmap="gray")
            axes[2].set_title("Unsharp")

            for ax in axes:
                ax.axis("off")

        elif view == "Image + Model + Residuals":

            fig, axes = plt.subplots(1, 3, figsize=(15,5))

            axes[0].imshow(image)
            axes[0].set_title("Image")

            axes[1].imshow(model)
            axes[1].set_title("Model")

            axes[2].imshow(residual)
            axes[2].set_title("Residuals")

            for ax in axes:
                ax.axis("off")

        elif view == "6-view":

            fig, axes = plt.subplots(2, 3, figsize=(15,10))

            axes = axes.ravel()

            axes[0].imshow(image)
            axes[0].set_title("Image")

            axes[1].imshow(model)
            axes[1].set_title("Model")

            axes[2].imshow(residual)
            axes[2].set_title("Residuals")

            axes[3].imshow(rgb, origin="lower")
            axes[3].set_title("RGB")

            axes[4].imshow(gray, origin="lower", cmap="gray")
            axes[4].set_title("Grayscale")

            axes[5].imshow(unsharp, origin="lower", cmap="gray")
            axes[5].set_title("Unsharp")

            for ax in axes:
                ax.axis("off")

        # Mistake label
        fig.suptitle(f"{row['true_class']} mistaken for {row['your_class']}", fontsize=14)
        plt.show()

        # Coordinates
        print(f"RA = {row['RA']:.4f}")
        print(f"DEC = {row['DEC']:.4f}")

In [5]:
# Connect widgets
index_box.observe(display_flagged_galaxy, names="value")
view_selector.observe(display_flagged_galaxy, names="value")
stretch_selector.observe(display_flagged_galaxy, names="value")

display(index_box, view_selector, stretch_selector, image_out)

# Initial display
display_flagged_galaxy()

IntText(value=0, description='Galaxy')

Dropdown(description='View', index=5, options=('RGB', 'Lupton RGB', 'Grayscale', 'Unsharp Mask', 'RGB + Graysc…

Dropdown(description='Stretch', options=('asinh', 'log', 'linear'), value='asinh')

Output()